# Phase 4 GPU benchmark

Benchmark Full-FP16, H2O và VECTOR trên GPU. Notebook đo prefill latency, decode throughput, inter-token latency, CUDA peak memory, persistent KV bytes và token đầu.

In [ ]:
import datetime, hashlib, json, os, random, sys, time
from pathlib import Path
import torch
import transformers
from transformers import AutoConfig, AutoModelForCausalLM

REPO_URL = os.environ.get('PROJECTIII_REPO_URL', 'https://github.com/Dung34/vector-kv-cache.git')
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    import subprocess
    REPO_DIR = ROOT / 'vector-kv-cache'
    if not (REPO_DIR / 'src').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    ROOT = REPO_DIR
for mod_name in list(sys.modules):
    if mod_name == 'src' or mod_name.startswith('src.'):
        del sys.modules[mod_name]
sys.path.insert(0, str(ROOT))
from src.baselines.full_cache import FullKVCacheWrapper
from src.baselines.h2o_cache import H2OKVCache
from src.baselines.h2o_hook import H2OAttentionHookManager
from src.vector_cache.vector_cache import VectorKVCache
from src.vector_cache.vector_hook import VectorAttentionHookManager
import inspect
assert 'target_budget' in inspect.signature(H2OKVCache.__init__).parameters, 'Loaded repository is stale: push/supply the Phase 2b+ source before running this notebook'

SEED = 42
CONTEXT_LENGTH = 256
DECODE_STEPS = 30
NONCE = 'REPLACE_WITH_TIMESTAMP_GPU_RUN'
DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print({'nonce': NONCE, 'timestamp': datetime.datetime.now().isoformat(), 'python': sys.version.split()[0], 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': torch.cuda.get_device_name(DEVICE), 'seed': SEED})

In [ ]:
config = AutoConfig.for_model('qwen2', vocab_size=1000, hidden_size=128, intermediate_size=256, num_hidden_layers=4, num_attention_heads=8, num_key_value_heads=2, max_position_embeddings=4096, use_cache=True)
config._attn_implementation = 'eager'
base = AutoModelForCausalLM.from_config(config).eval().to(DEVICE)
state = {key: value.detach().clone() for key, value in base.state_dict().items()}
prompt = torch.randint(0, config.vocab_size, (1, CONTEXT_LENGTH), device=DEVICE)
head_dim = config.hidden_size // config.num_attention_heads
w_ols = torch.eye(head_dim, device=DEVICE).view(1, 1, head_dim, head_dim).expand(config.num_hidden_layers, config.num_key_value_heads, head_dim, head_dim).contiguous()
print({'context_length': prompt.shape[-1], 'hidden_size': config.hidden_size, 'layers': config.num_hidden_layers, 'kv_heads': config.num_key_value_heads, 'head_dim': head_dim})

In [ ]:
def run_variant(name, cache, manager_cls):
    model = AutoModelForCausalLM.from_config(config).eval().to(DEVICE)
    model.load_state_dict(state)
    manager = None
    if manager_cls is not None:
        manager = manager_cls(model=model, kv_cache=cache); manager.register_hooks()
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(DEVICE); torch.cuda.synchronize(DEVICE)
    with torch.inference_mode():
        start = time.perf_counter(); output = model(prompt, past_key_values=cache, use_cache=True); torch.cuda.synchronize(DEVICE)
    prefill_ms = (time.perf_counter() - start) * 1000
    generated, decode_times = [], []
    for _ in range(DECODE_STEPS):
        token = output.logits[:, -1:].argmax(dim=-1); generated.append(token.item())
        torch.cuda.synchronize(DEVICE); start = time.perf_counter()
        with torch.inference_mode(): output = model(token, past_key_values=cache, use_cache=True)
        torch.cuda.synchronize(DEVICE); decode_times.append(time.perf_counter() - start)
    if manager is not None: manager.remove_hooks()
    result = {'variant': name, 'context_length': CONTEXT_LENGTH, 'decode_tokens': DECODE_STEPS, 'prefill_ms': round(prefill_ms, 3), 'decode_mean_ms': round(sum(decode_times) / len(decode_times) * 1000, 3), 'decode_tokens_per_sec': round(DECODE_STEPS / sum(decode_times), 3), 'inter_token_latency_ms': round(sum(decode_times) / len(decode_times) * 1000, 3), 'persistent_kv_bytes': cache.get_memory_bytes(), 'peak_allocated_bytes': torch.cuda.max_memory_allocated(DEVICE), 'peak_reserved_bytes': torch.cuda.max_memory_reserved(DEVICE), 'physical_len': [cache._physical_len(i) if hasattr(cache, '_physical_len') else cache.get_seq_length(i) for i in range(config.num_hidden_layers)], 'generated': generated}
    print(result); return result

variants = [('full_fp16', FullKVCacheWrapper(), None), ('h2o', H2OKVCache(target_budget=128, max_physical_budget=256, min_recent_tokens=16), H2OAttentionHookManager), ('vector', VectorKVCache(target_budget=128, max_physical_budget=256, compression_ratio=0.5, approx_ratio=0.1, min_recent_tokens=16, w_ols=w_ols), VectorAttentionHookManager)]
results = [run_variant(name, cache, manager) for name, cache, manager in variants]

In [ ]:
baseline = results[0]['generated'][0]
for result in results[1:]:
    result['first_token_matches_full'] = result['generated'][0] == baseline
    print({'variant': result['variant'], 'expected_first_token': baseline, 'actual_first_token': result['generated'][0], 'verdict': 'PASS' if result['first_token_matches_full'] else 'INFO'})
report_dir = ROOT / 'reports'; report_dir.mkdir(exist_ok=True)
with open(report_dir / 'phase4_gpu_benchmark.json', 'w', encoding='utf-8') as handle: json.dump({'nonce': NONCE, 'results': results}, handle, indent=2)
with open(report_dir / 'phase4_gpu_benchmark.md', 'w', encoding='utf-8') as handle:
    handle.write('| variant | prefill_ms | tok/s | persistent_kv_bytes | peak_allocated_bytes | peak_reserved_bytes | first_token_matches_full |\n|---|---:|---:|---:|---:|---:|---|\n')
    for item in results: handle.write(f"| {item['variant']} | {item['prefill_ms']} | {item['decode_tokens_per_sec']} | {item['persistent_kv_bytes']} | {item['peak_allocated_bytes']} | {item['peak_reserved_bytes']} | {item.get('first_token_matches_full', True)} |\n")
print('reports_written', report_dir / 'phase4_gpu_benchmark.json', report_dir / 'phase4_gpu_benchmark.md')